# Song Lyrics Generator

Pipeline overview:
1. Load a `.wav` file from `sample_data/`
2. Transcribe with Sarvam Speech-to-Text (`saaras:v3`)
3. Save lyrics text to `outputs/song_lyrics.txt`


In [ ]:
%pip install -r requirements.txt


## Setup


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
SAMPLE_DIR = Path("sample_data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## Transcribe audio


In [ ]:
def transcribe_audio(file_path: Path, language_code: str = "hi-IN") -> str:
    """Transcribe a local audio file with saaras:v3 (batch REST)."""
    with file_path.open("rb") as audio_file:
        response = client.speech_to_text.transcribe(
            file=audio_file,
            model="saaras:v3",
            mode="transcribe",
            language_code=language_code,
        )
    if hasattr(response, "transcript"):
        return response.transcript or ""
    if isinstance(response, dict):
        return response.get("transcript", "") or ""
    return str(response)


## Run


In [ ]:
# Place a .wav file in sample_data/ and update the name below.
AUDIO_PATH = SAMPLE_DIR / "song.wav"
LANGUAGE_CODE = "hi-IN"

if not AUDIO_PATH.exists():
    raise FileNotFoundError(
        f"Missing {AUDIO_PATH}. Add a .wav file under sample_data/ first."
    )

lyrics = transcribe_audio(AUDIO_PATH, language_code=LANGUAGE_CODE)
print(lyrics)

out_path = OUTPUT_DIR / "song_lyrics.txt"
out_path.write_text(lyrics, encoding="utf-8")
print(f"Wrote {out_path}")
